# Notebook 04: Model Training (`sobelv5`, three-stage training plan)

### English

**The idea.** One conditional DCGAN, trained in three stages that each have a single job. Splitting training
this way means the recipe can be aggressive where that helps (finding face structure fast) and cautious where
that helps (holding onto structure while sharpening), instead of using one compromise recipe for the whole
run.

| Stage | Epochs | Job | Recipe |
|---|---|---|---|
| `stage1` | 150 | Find the face structure fast | Equal learning rate for G and D, no extra regularization, no EMA |
| `stage2` | 200 | Refine slowly and safely | TTUR (D's LR held to a third of G's) + instance noise annealed over the first 60% + DiffAugment (translation only) + EMA (0.95) + a late LR taper |
| `stage3` | up to 100 | Sharpen and hold | Very low LR, instance noise off, DiffAugment translation, EMA (0.90), a mode-seeking diversity term (`ms_weight=0.2`), a LR floor that stops short of fully freezing (`lr_floor=0.3`), early stop on detected collapse, best-checkpoint saving |

Stage2's 200-epoch length gives the TTUR-restrained recipe a long runway to settle rather than being cut
short mid-refinement. Stage3's mode-seeking weight is set moderately (0.2) so it discourages many-noise-values
collapsing onto one face without fighting too hard against the sharpening objective, and its LR floor (0.3)
keeps a little learning signal alive all the way to the end of the stage instead of the run coasting to a
near-stop. Stage3 also never trusts itself blindly: it stops early after two consecutive readings that look
like collapse, and it keeps a scored `best/` checkpoint updated throughout, so the exported model does not
have to be whatever the final epoch happened to produce.

Every stage uses the **identical architecture** -- stage2 loads stage1's weights, stage3 loads stage2's EMA
weights, and no layer shape ever changes between them. Both dog and cat are trained with **no per-class
weighting**: the loss treats every image the same regardless of species, so neither class is deliberately
favored.

**Naming.** This model is **`sobelv5`** and writes to `models/sobelv5/`.

**Monitoring.** Sample grids show 16 images per class (32 total), laid out at exactly 1920x1080 with
class names in the left margin so no label is ever covered by an image. Per-class `struct` (shape diversity),
`color` (palette spread) and `sharp` (edge strength vs. a real-image reference) metrics are logged every
checkpoint. Three kinds of weights are saved per stage: a rolling checkpoint for resuming, permanent
`snapshots/` every `snapshot_every` epochs that are never overwritten, and a scored `best/` checkpoint that
tracks the sharpest-while-still-diverse model seen so far.

### Tiếng Việt

**Ý tưởng.** Một conditional DCGAN, được train qua ba stage, mỗi stage một nhiệm vụ riêng. Tách train ra
như vậy để công thức có thể mạnh dạn ở chỗ cần mạnh dạn (tìm cấu trúc khuôn mặt nhanh) và thận trọng ở chỗ
cần thận trọng (giữ cấu trúc trong lúc làm sắc nét), thay vì dùng một công thức thỏa hiệp cho cả quá trình.

| Stage | Epoch | Nhiệm vụ | Công thức |
|---|---|---|---|
| `stage1` | 150 | Tìm cấu trúc khuôn mặt nhanh | LR bằng nhau cho G và D, không thêm regularization, không EMA |
| `stage2` | 200 | Tinh chỉnh chậm và an toàn | TTUR (LR của D chỉ bằng 1/3 của G) + instance noise giảm dần trong 60% đầu + DiffAugment (chỉ translation) + EMA (0.95) + taper LR về cuối |
| `stage3` | tối đa 100 | Làm sắc nét và giữ vững | LR rất thấp, tắt instance noise, DiffAugment translation, EMA (0.90), số hạng đa dạng mode-seeking (`ms_weight=0.2`), sàn LR không đóng băng hoàn toàn (`lr_floor=0.3`), tự dừng khi phát hiện collapse, lưu checkpoint tốt nhất |

Độ dài 200 epoch của stage2 cho công thức bị TTUR kìm hãm một quãng đường đủ dài để ổn định, thay vì bị
cắt ngang giữa chừng tinh chỉnh. Trọng số mode-seeking của stage3 đặt ở mức vừa phải (0.2) để ngăn nhiều
giá trị noise dồn về cùng một khuôn mặt mà không cạnh tranh quá mạnh với mục tiêu làm sắc nét, còn sàn LR
(0.3) giữ cho một chút tín hiệu học vẫn còn sống đến hết stage, thay vì lần chạy gần như "đứng yên" ở cuối.
Stage3 cũng không tin tưởng mù quáng vào chính nó: nó tự dừng sau hai lần đo liên tiếp trông giống collapse,
và luôn giữ một checkpoint `best/` có chấm điểm cập nhật xuyên suốt, để model export ra không nhất thiết
là bất cứ thứ gì epoch cuối cùng tạo ra.

Mọi stage đều dùng **kiến trúc y hệt nhau** -- stage2 nạp trọng số của stage1, stage3 nạp trọng số EMA của
stage2, và không hình dạng layer nào thay đổi giữa các stage. Cả chó và mèo đều được train **không có
trọng số ưu tiên loài nào**: loss đối xử với mọi ảnh như nhau bất kể loài, nên không loài nào được ưu ái
có chủ đích.

**Tên gọi.** Model này tên là **`sobelv5`**, ghi vào `models/sobelv5/`.

**Theo dõi.** Ảnh mẫu hiển thị 16 ảnh mỗi loài (32 ảnh tổng), sắp xếp đúng khung 1920x1080 với tên loài
đặt ở lề trái để nhãn không bao giờ bị ảnh che. Các chỉ số riêng từng loài `struct` (đa dạng hình dạng),
`color` (độ trải màu) và `sharp` (độ sắc nét so với ảnh thật) được ghi log ở mỗi checkpoint. Ba loại trọng
số được lưu mỗi stage: checkpoint quay vòng để resume, `snapshots/` vĩnh viễn mỗi `snapshot_every` epoch
không bao giờ bị ghi đè, và checkpoint `best/` có chấm điểm theo dõi model sắc nét nhất mà vẫn còn đa dạng
tính đến thời điểm đó.

## 1. Environment Setup

### English
Import TensorFlow/Keras, confirm a GPU is attached, and fix every random seed so runs are comparable.

### Tiếng Việt
Import TensorFlow/Keras, kiểm tra có GPU, và cố định mọi seed ngẫu nhiên để các lần chạy so sánh được với
nhau.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import random

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU(s) available: {[g.name for g in gpus]}")
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> GPU before training.")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

## 2. Mount Google Drive and Locate the Processed Dataset

### English
All weights, sample grids and history files live under `MyDrive/dog-gan-project` so a disconnected Colab
session never loses progress.

### Tiếng Việt
Toàn bộ trọng số, ảnh mẫu và file lịch sử nằm trong `MyDrive/dog-gan-project` để khi Colab ngắt kết nối
vẫn không mất tiến độ.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/dog-gan-project")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "04_training"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PROCESSED.exists(), f"Processed data not found at {DATA_PROCESSED} -- run 03_preprocessing.ipynb and upload its output first."
print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {DATA_PROCESSED}")

## 3. Load Manifests and Configure Hyperparameters

### English
Shared settings first (image size, batch size, noise dimension) -- these are identical across all three
stages, because the three stages share one architecture and one set of weight shapes.

`MODEL_SUBDIR = "sobelv5"` and each stage writes into its own subfolder:

```
models/sobelv5/stage1/   stage2/   stage3/
    generator.weights.h5          <- rolling latest, overwritten every CHECKPOINT_EVERY epochs
    discriminator.weights.h5
    generator_ema.weights.h5      <- stage2/stage3 only
    history.csv                   <- one row per epoch, never overwritten
    snapshots/generator_e050.weights.h5 ...   <- kept forever, this is the rollback safety net
    best/generator_best.weights.h5            <- highest-scoring checkpoint seen in this stage
```

The `snapshots/` folder is kept separate from the rolling checkpoint on purpose: the rolling checkpoint is
overwritten every `CHECKPOINT_EVERY` epochs, so on its own it offers no way back if a later epoch turns out
worse. A permanent copy every `snapshot_every` epochs means there is always an earlier version to fall back
to.

### Tiếng Việt
Các tham số dùng chung trước (kích thước ảnh, batch, số chiều noise) -- ba stage dùng y hệt nhau, vì cả ba
dùng chung một kiến trúc và cùng một hình dạng trọng số.

`MODEL_SUBDIR = "sobelv5"` và mỗi stage ghi vào thư mục riêng của nó (xem sơ đồ ở trên).

Thư mục `snapshots/` được tách riêng khỏi checkpoint quay vòng có chủ đích: checkpoint quay vòng bị ghi
đè mỗi `CHECKPOINT_EVERY` epoch, nên tự nó không có đường quay lại nếu epoch sau tệ hơn. Một bản vĩnh viễn
mỗi `snapshot_every` epoch nghĩa là luôn có một bản trước đó để lùi về khi cần.

In [ ]:
import pandas as pd

train_manifest = pd.read_csv(DATA_PROCESSED / "train_manifest.csv")
val_manifest = pd.read_csv(DATA_PROCESSED / "val_manifest.csv")

SPECIES_TO_LABEL = {"dog": 0, "cat": 1}
LABEL_TO_SPECIES = {v: k for k, v in SPECIES_TO_LABEL.items()}
NUM_CLASSES = len(SPECIES_TO_LABEL)

IMAGE_SIZE = 128
NOISE_DIM = 100
EMBEDDING_DIM = 50
BATCH_SIZE = 128
CHECKPOINT_EVERY = 5
MODEL_SUBDIR = "sobelv5"

BETA_1 = 0.5
REAL_LABEL_SMOOTHING = 0.9  # one-sided label smoothing on the "real" target

# Collapse guard (used by stage 3's early stop and by the warnings in every stage)
HEALTH_TARGET = 0.85        # struct diversity we would like to hold, as a fraction of the real-image reference
COLLAPSE_STOP_RATIO = 0.60  # below this fraction of the real reference = treat as collapsing
COLLAPSE_PATIENCE = 2       # consecutive checkpoint evaluations below the ratio before stage 3 stops

print(f"Train images: {len(train_manifest)} -- {train_manifest['species'].value_counts().to_dict()}")
print(f"Val images  : {len(val_manifest)} -- {val_manifest['species'].value_counts().to_dict()}")
print(f"Species -> label mapping: {SPECIES_TO_LABEL}")
print(f"Steps per epoch (approx): {len(train_manifest) // BATCH_SIZE}")

### 3b. The three stage recipes

### English
Every knob that differs between stages lives in this one dictionary. Reading it top to bottom is the
whole training plan.

- `gen_lr` / `disc_lr` -- stage 1 keeps them equal (the proven fast-structure recipe). Stage 2 drops the
  Discriminator to a third of the Generator's rate (TTUR), because `disc_acc` at 0.90-0.95 says D is winning.
  Stage 3 drops both far down for fine polishing.
- `instance_noise` -- Gaussian noise added to whatever D sees, annealed to zero over the first 60% of the
  stage. It softens D's decision boundary. Stage 3 sets it to 0 **on purpose**: noise is what lets blurry
  output survive, and stage 3's job is sharpness.
- `diffaug` -- DiffAugment, applied identically to real and fake images before D. `translation` only:
  it stops D memorizing the real set without touching colour, which keeps the Generator's palette free to
  vary naturally.
- `ema_decay` -- per-epoch exponential moving average of the Generator's weights. The EMA copy is what gets
  sampled and exported; it is visibly cleaner than the raw Generator and much less prone to epoch-to-epoch
  wobble. 0.95 = roughly a 20-epoch averaging window.
- `ms_weight` -- mode-seeking term (stage 3 only). Within one batch it pairs up samples **of the same class**
  and rewards "different noise vector -> different image", which directly discourages many different noise
  values from collapsing onto one face.
- `lr_decay_from` / `lr_floor` -- LR stays flat until this fraction of the stage, then falls linearly to
  `lr_floor` x its base value. These are **fractions**, so lengthening a stage automatically pushes the
  absolute epoch where decay starts (and where the floor is reached) further out -- no separate edit needed.
- `early_stop` -- stage 3 only. Two consecutive collapse readings and the stage stops, leaving `best/` intact.

**Why stage2 gets 200 epochs and stage3 gets these particular numbers:**

- `stage2.epochs = 200` gives the TTUR-restrained refinement recipe real room to work: instance noise has
  time to anneal out gradually, the LR taper only begins at epoch 140, and `disc_acc` has a long stretch to
  stay comfortably controlled while the Generator keeps closing the gap.
- `stage3.epochs = 100`, `ms_weight = 0.2`, `lr_floor = 0.3` are chosen together. A hundred epochs is enough
  runway for a sharpening pass to do real work; `ms_weight = 0.2` is deliberately moderate -- strong enough to
  keep pushing back against many-noise-values collapsing onto one face, but not so strong that it fights the
  sharpening objective for the whole stage; `lr_floor = 0.3` keeps roughly a third of the base learning rate
  alive at the very end, so the last epochs still carry some ability to improve rather than coasting.
- `stage1 = 150` epochs is enough for the fast plain-DCGAN recipe to fully find face structure for both
  species before handing off to stage2.

### Tiếng Việt
Mọi tham số khác nhau giữa các stage đều nằm trong dictionary này. Đọc từ trên xuống là thấy cả kế hoạch
train.

- `gen_lr` / `disc_lr` -- stage 1 để bằng nhau (công thức tìm cấu trúc nhanh đã chứng minh). Stage 2 hạ D
  xuống còn 1/3 của G (TTUR), vì `disc_acc` 0.90-0.95 nghĩa là D đang thắng. Stage 3 hạ cả hai xuống rất
  thấp để đánh bóng.
- `instance_noise` -- nhiễu Gauss thêm vào ảnh mà D nhìn thấy, giảm dần về 0 trong 60% đầu stage, làm mềm
  ranh giới quyết định của D. Stage 3 đặt = 0 **có chủ ý**: chính nhiễu là thứ cho phép ảnh mờ tồn tại, mà
  nhiệm vụ của stage 3 là làm rõ nét.
- `diffaug` -- DiffAugment, áp dụng y hệt nhau cho ảnh thật và ảnh giả trước khi vào D. Chỉ dùng
  `translation`: nó ngăn D học thuộc tập thật mà không đụng đến màu, giữ cho bảng màu của Generator được
  tự do biến đổi tự nhiên.
- `ema_decay` -- trung bình trượt theo epoch của trọng số G. Bản EMA là bản được lấy mẫu và export; nó
  sạch hơn bản thô và ít dao động giữa các epoch hơn nhiều. 0.95 = cửa sổ trung bình khoảng 20 epoch.
- `ms_weight` -- số hạng mode-seeking (chỉ stage 3). Trong một batch nó ghép cặp các mẫu **cùng loài** và
  thưởng cho "noise khác nhau → ảnh khác nhau", trực tiếp ngăn nhiều giá trị noise khác nhau dồn về cùng
  một khuôn mặt.
- `lr_decay_from` / `lr_floor` -- LR giữ nguyên đến mốc này của stage, rồi giảm tuyến tính xuống còn
  `lr_floor` lần. Đây là **tỷ lệ phần trăm**, nên kéo dài một stage sẽ tự động đẩy mốc epoch tuyệt đối của
  nó ra xa hơn -- không cần sửa riêng.
- `early_stop` -- chỉ stage 3. Hai lần đo liên tiếp báo collapse là stage dừng, `best/` vẫn còn nguyên.

**Vì sao stage2 chạy 200 epoch và stage3 dùng đúng các con số này:**

- `stage2.epochs = 200` cho công thức tinh chỉnh bị TTUR kìm hãm đủ không gian để làm việc: instance noise
  có thời gian giảm dần từ từ, taper LR chỉ bắt đầu ở epoch 140, và `disc_acc` có một quãng dài để giữ ổn
  định trong khi Generator tiếp tục thu hẹp khoảng cách.
- `stage3.epochs = 100`, `ms_weight = 0.2`, `lr_floor = 0.3` được chọn cùng nhau. Một trăm epoch là đủ để
  một pha làm sắc nét thực sự có tác dụng; `ms_weight = 0.2` được chọn ở mức vừa phải có chủ đích -- đủ
  mạnh để tiếp tục đẩy lùi việc nhiều giá trị noise dồn về cùng một khuôn mặt, nhưng không mạnh đến mức
  cạnh tranh với mục tiêu làm sắc nét suốt cả stage; `lr_floor = 0.3` giữ khoảng một phần ba LR gốc còn
  sống đến tận cuối, để những epoch cuối vẫn còn khả năng cải thiện thay vì chỉ "trôi".
- `stage1 = 150` epoch là đủ để công thức DCGAN thuần chạy nhanh tìm ra cấu trúc khuôn mặt đầy đủ cho cả
  hai loài trước khi bàn giao cho stage2.

In [ ]:
STAGES = {
    "stage1": dict(
        epochs=150,
        gen_lr=2e-4, disc_lr=2e-4,      # equal LR -- the proven fast-structure recipe
        instance_noise=0.0, diffaug="", ema_decay=0.0, ms_weight=0.0,
        lr_decay_from=1.0, lr_floor=1.0,  # constant LR for the whole stage
        init_from=None, init_from_ema=False,
        snapshot_every=25, early_stop=False,
    ),
    "stage2": dict(
        epochs=200,  # a long, safe refinement runway under TTUR
        gen_lr=1.5e-4, disc_lr=5e-5,    # TTUR: D deliberately slowed down
        instance_noise=0.05, diffaug="translation", ema_decay=0.95, ms_weight=0.0,
        lr_decay_from=0.7, lr_floor=0.4,  # decay starts at epoch 140, floor reached at epoch 200
        init_from="stage1", init_from_ema=False,
        snapshot_every=25, early_stop=False,
    ),
    "stage3": dict(
        epochs=100,  # enough runway for a sharpening pass to do real work
        gen_lr=5e-5, disc_lr=2.5e-5,    # very low: polish, do not re-learn
        instance_noise=0.0,             # off on purpose -- noise protects blur, we want sharpness
        diffaug="translation", ema_decay=0.90,
        ms_weight=0.2,   # moderate: pushes back on collapse without overpowering sharpening
        lr_decay_from=0.3, lr_floor=0.3,  # keeps some learning signal alive through the end of the stage
        init_from="stage2", init_from_ema=True,
        snapshot_every=10, early_stop=True,
    ),
}

for key, cfg in STAGES.items():
    print(f"{key}: {cfg['epochs']} epochs, G lr={cfg['gen_lr']:.1e}, D lr={cfg['disc_lr']:.1e}, "
          f"noise={cfg['instance_noise']}, diffaug='{cfg['diffaug']}', ema={cfg['ema_decay']}, "
          f"ms={cfg['ms_weight']}, lr_floor={cfg['lr_floor']}, early_stop={cfg['early_stop']}")

## 4. Copy the Dataset Locally, Then Build the `tf.data` Input Pipeline

### English
Reading 128x128 JPEGs straight from Drive is the slowest part of a Colab epoch, so the dataset is copied to
the local VM disk once per session. Augmentation is horizontal flip only -- everything stronger is applied
later, inside the Discriminator's input path, where DiffAugment can be turned on and off per stage.

### Tiếng Việt
Đọc JPEG 128x128 trực tiếp từ Drive là phần chậm nhất của một epoch trên Colab, nên dataset được chép ra
ổ đĩa cục bộ của VM một lần mỗi phiên. Augment chỉ lật ngang -- mọi thứ mạnh hơn đều áp dụng sau, ngay
trước đầu vào của Discriminator, nơi DiffAugment có thể bật/tắt theo từng stage.

In [ ]:
import shutil

LOCAL_IMAGES_DIR = Path("/content/data_local/images_128")

if not LOCAL_IMAGES_DIR.exists():
    LOCAL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    print("Copying images_128 to local disk (first time this session)...")
    shutil.copytree(DATA_PROCESSED / "images_128", LOCAL_IMAGES_DIR, dirs_exist_ok=True)
    print(f"Copied {len(list(LOCAL_IMAGES_DIR.glob('*')))} files.")
else:
    print("Local copy of images_128 already present, skipping copy.")

In [ ]:
def load_image(filepath, label):
    raw = tf.io.read_file(filepath)
    image = tf.io.decode_jpeg(raw, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0  # normalize to [-1, 1]
    return image, label


def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    return image, label


def build_dataset(manifest_df, images_dir, batch_size, shuffle=True, augment_flag=True):
    filenames = [Path(fp.replace("\\", "/")).name for fp in manifest_df["filepath"]]
    filepaths = [str(images_dir / name) for name in filenames]
    labels = manifest_df["species"].map(SPECIES_TO_LABEL).astype("int32").to_numpy()

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_flag:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = build_dataset(train_manifest, LOCAL_IMAGES_DIR, BATCH_SIZE, shuffle=True, augment_flag=True)
val_ds = build_dataset(val_manifest, LOCAL_IMAGES_DIR, BATCH_SIZE, shuffle=False, augment_flag=False)

print(f"train_ds element spec: {train_ds.element_spec}")

In [ ]:
import matplotlib.pyplot as plt

sample_images, sample_labels = next(iter(train_ds))
sample_images_disp = ((sample_images.numpy() + 1.0) / 2.0).clip(0, 1)

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(sample_images_disp[i])
    ax.set_title(LABEL_TO_SPECIES[int(sample_labels[i])], fontsize=9)
    ax.axis("off")
fig.suptitle("Real training batch (after normalization and flip augmentation)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_real_batch.png", dpi=150)
plt.show()

## 5. Discriminator Auxiliary Channel (Sobel)

### English
The Discriminator sees a fifth input channel: the Sobel edge magnitude of the image it is judging, computed
inside the model. It gives D an explicit view of edge quality, which pushes the Generator toward crisper
outlines rather than smooth blobs. Sobel is the only auxiliary channel still in use -- the "Default" and
"+Laplacian" variants were retired earlier.

### Tiếng Việt
Discriminator nhận thêm một kênh đầu vào thứ năm: độ lớn cạnh Sobel của chính ảnh nó đang đánh giá, tính
ngay bên trong model. Kênh này cho D một góc nhìn rõ ràng về chất lượng cạnh, từ đó đẩy Generator về phía
đường nét sắc hơn thay vì các mảng mờ. Sobel là kênh phụ duy nhất còn dùng -- bản "Default" và
"+Laplacian" đã bỏ từ trước.

In [ ]:
def _per_image_normalize(magnitude):
    """Min-max normalize a (B, H, W, 1) magnitude map to [-1, 1], per image."""
    min_val = tf.reduce_min(magnitude, axis=[1, 2, 3], keepdims=True)
    max_val = tf.reduce_max(magnitude, axis=[1, 2, 3], keepdims=True)
    normalized = (magnitude - min_val) / (max_val - min_val + 1e-8)
    return normalized * 2.0 - 1.0


def sobel_channel(images):
    """images: (B, H, W, 3) float32 in [-1, 1]. Returns (B, H, W, 1) float32 in [-1, 1]."""
    gray = tf.image.rgb_to_grayscale((images + 1.0) / 2.0)
    edges = tf.image.sobel_edges(gray)  # (B, H, W, 1, 2) -- [dy, dx]
    magnitude = tf.sqrt(tf.reduce_sum(tf.square(edges), axis=-1) + 1e-8)  # (B, H, W, 1)
    return _per_image_normalize(magnitude)


sobel_demo = sobel_channel(sample_images).numpy()

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for col in range(6):
    axes[0, col].imshow(sample_images_disp[col])
    axes[0, col].set_title("RGB", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(sobel_demo[col, :, :, 0], cmap="gray")
    axes[1, col].set_title("Sobel channel", fontsize=9)
    axes[1, col].axis("off")
fig.suptitle("Discriminator auxiliary channel (Sobel)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_auxiliary_channel.png", dpi=150)
plt.show()

## 6. Species-Conditioning Channel

### English
Conditioning is symmetric: the Generator receives the species as a learned embedding concatenated to the
noise vector, and the Discriminator receives it as a constant-valued image channel (-1 for dog, +1 for cat),
so D can judge "is this a convincing *cat*" rather than just "is this a convincing animal". This shared,
single-network design is also why dog and cat quality can drift apart -- they are not two separate models,
just one network conditioned on this channel plus the embedding.

### Tiếng Việt
Điều kiện hóa đối xứng hai phía: Generator nhận loài dưới dạng embedding học được, nối vào vector noise;
Discriminator nhận dưới dạng một kênh ảnh giá trị hằng số (-1 cho chó, +1 cho mèo), nhờ vậy D đánh giá
được "đây có phải là một con *mèo* thuyết phục không" chứ không chỉ "có phải một con vật thuyết phục
không". Chính thiết kế một mạng dùng chung này cũng là lý do chất lượng chó và mèo có thể lệch nhau --
đây không phải hai model riêng, chỉ là một mạng được điều kiện hóa bằng kênh này cộng embedding.

In [ ]:
def species_channel(labels, size=IMAGE_SIZE):
    """labels: (B,) int32 in {0, 1}. Returns (B, size, size, 1) float32: -1 for dog, +1 for cat."""
    value = tf.cast(labels, tf.float32) * 2.0 - 1.0
    value = tf.reshape(value, [-1, 1, 1, 1])
    batch_size = tf.shape(labels)[0]
    return tf.ones([batch_size, size, size, 1], dtype=tf.float32) * value


species_demo = species_channel(sample_labels).numpy()
dog_idx = int(np.argmax(sample_labels.numpy() == 0))
cat_idx = int(np.argmax(sample_labels.numpy() == 1))

fig, axes = plt.subplots(1, 2, figsize=(6, 3.2))
axes[0].imshow(species_demo[dog_idx, :, :, 0], cmap="gray", vmin=-1, vmax=1)
axes[0].set_title(f"Species channel -- dog (value={species_demo[dog_idx,0,0,0]:.0f})", fontsize=9)
axes[0].axis("off")
axes[1].imshow(species_demo[cat_idx, :, :, 0], cmap="gray", vmin=-1, vmax=1)
axes[1].set_title(f"Species channel -- cat (value={species_demo[cat_idx,0,0,0]:.0f})", fontsize=9)
axes[1].axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_species_channel.png", dpi=150)
plt.show()

## 7. Generator Architecture

### English
`Dense -> 8x8x256`, then four `Conv2DTranspose` blocks to 128x128, `tanh` output.
**Do not edit this cell between stages** -- stage 2 and stage 3 load stage 1's weight file, and any change
to layer shapes makes that load fail.

### Tiếng Việt
`Dense → 8x8x256`, rồi bốn khối `Conv2DTranspose` lên 128x128, đầu ra `tanh`.
**Không được sửa cell này giữa các stage** -- stage 2 và 3 nạp file trọng số của stage 1, chỉ cần đổi hình
dạng layer là việc nạp sẽ lỗi.

In [ ]:
from tensorflow.keras import layers, Model


def build_generator(noise_dim=NOISE_DIM, num_classes=NUM_CLASSES, embedding_dim=EMBEDDING_DIM, name="generator"):
    noise_input = layers.Input(shape=(noise_dim,), name="noise")
    label_input = layers.Input(shape=(), dtype="int32", name="label")

    label_embed = layers.Embedding(num_classes, embedding_dim)(label_input)
    x = layers.Concatenate()([noise_input, label_embed])

    x = layers.Dense(8 * 8 * 256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((8, 8, 256))(x)

    x = layers.Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False)(x)  # 16x16
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False)(x)  # 32x32
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(32, 4, strides=2, padding="same", use_bias=False)(x)  # 64x64
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    output = layers.Conv2DTranspose(3, 4, strides=2, padding="same", activation="tanh")(x)  # 128x128

    return Model([noise_input, label_input], output, name=name)


_g = build_generator()
_g.summary()
print(f"\nTotal Generator parameters: {_g.count_params():,}")
del _g

## 8. Discriminator Architecture

### English
Four strided `Conv2D` blocks over the 5-channel input (RGB + species + Sobel), BatchNorm, Dropout(0.3) on
the first two blocks, single logit out. No Spectral Normalization and no minibatch-stddev -- D is restrained
in stage 2 and stage 3 by *training-time* means only (lower LR, instance noise, DiffAugment), none of which
touch weight shapes, so the same architecture carries cleanly across all three stages.

### Tiếng Việt
Bốn khối `Conv2D` stride trên đầu vào 5 kênh (RGB + loài + Sobel), BatchNorm, Dropout(0.3) ở hai khối đầu,
đầu ra một logit. Không Spectral Normalization, không minibatch-stddev -- D chỉ bị kìm hãm ở stage 2 và
stage 3 bằng các cách *lúc train* (LR thấp hơn, instance noise, DiffAugment), không cách nào đụng đến hình
dạng trọng số, nên cùng một kiến trúc mang xuyên suốt cả ba stage.

In [ ]:
def build_discriminator(name="discriminator"):
    image_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
    label_input = layers.Input(shape=(), dtype="int32", name="label")

    species_ch = layers.Lambda(lambda l: species_channel(l, size=IMAGE_SIZE), name="species_channel")(label_input)
    aux_ch = layers.Lambda(sobel_channel, name="sobel_channel")(image_input)
    x = layers.Concatenate(axis=-1)([image_input, species_ch, aux_ch])  # 5 channels

    x = layers.Conv2D(64, 4, strides=2, padding="same")(x)  # 64x64
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 4, strides=2, padding="same")(x)  # 32x32
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, 4, strides=2, padding="same")(x)  # 16x16
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(512, 4, strides=2, padding="same")(x)  # 8x8
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Flatten()(x)
    output = layers.Dense(1, name="logit")(x)

    return Model([image_input, label_input], output, name=name)


_d = build_discriminator()
_d.summary()
print(f"\nTotal Discriminator parameters: {_d.count_params():,}")
del _d

## 9. Loss Functions and Optimizers

### English
Standard non-saturating GAN loss with one-sided label smoothing (real target 0.9, not 1.0) -- it stops D
from becoming over-confident, which is the first step toward D overpowering G. `make_optimizers` takes the
two learning rates as arguments, because they differ per stage.

### Tiếng Việt
Hàm loss GAN non-saturating chuẩn, có label smoothing một phía (mục tiêu thật là 0.9 chứ không phải 1.0)
-- nó ngăn D trở nên quá tự tin, bước đầu tiên dẫn đến việc D áp đảo G. `make_optimizers` nhận hai
learning rate làm tham số, vì mỗi stage một khác.

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)


def discriminator_loss(real_logits, fake_logits):
    real_loss = bce(tf.ones_like(real_logits) * REAL_LABEL_SMOOTHING, real_logits)
    fake_loss = bce(tf.zeros_like(fake_logits), fake_logits)
    return real_loss + fake_loss


def generator_loss(fake_logits):
    return bce(tf.ones_like(fake_logits), fake_logits)


def make_optimizers(gen_lr, disc_lr):
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=gen_lr, beta_1=BETA_1)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=disc_lr, beta_1=BETA_1)
    return gen_optimizer, disc_optimizer


print("Loss functions and optimizer factory defined.")

## 10. Stage 2 / Stage 3 Stabilizers

### English
Four small tools, all of them training-time only, so weights stay loadable across stages.

- `rand_translation` (DiffAugment) shifts an image by up to 1/8 of its size, filling the edge with zeros.
  It is applied to real **and** fake images identically -- that symmetry is the whole trick: D cannot
  distinguish real from fake by the augmentation, so the augmentation cannot leak into the Generator's
  output the way a normal data augmentation would.
- `add_instance_noise` adds Gaussian noise to whatever D sees, with a std that anneals to zero.
- `update_ema` blends the live Generator into a shadow copy once per epoch.
- `set_lr` / `lr_factor` / `instance_noise_at` are the schedules.

### Tiếng Việt
Bốn công cụ nhỏ, tất cả chỉ tác động lúc train, nên trọng số vẫn nạp qua lại giữa các stage được.

- `rand_translation` (DiffAugment) dịch ảnh tối đa 1/8 kích thước, viền được lấp bằng 0. Nó áp dụng y hệt
  nhau cho ảnh thật **và** ảnh giả -- chính sự đối xứng đó là mấu chốt của kỹ thuật này: D không thể phân
  biệt thật/giả dựa vào phép augment, nên phép augment không thể rỉ vào đầu ra của Generator như augment
  dữ liệu thông thường.
- `add_instance_noise` thêm nhiễu Gauss vào ảnh mà D nhìn thấy, độ lệch chuẩn giảm dần về 0.
- `update_ema` trộn trọng số G hiện tại vào một bản sao bóng, mỗi epoch một lần.
- `set_lr` / `lr_factor` / `instance_noise_at` là các lịch giảm.

In [ ]:
def rand_translation(x, ratio=0.125):
    """DiffAugment translation: per-sample shift up to +/- ratio of the image size, zero-filled."""
    batch_size = tf.shape(x)[0]
    image_size = tf.shape(x)[1:3]
    shift = tf.cast(tf.cast(image_size, tf.float32) * ratio + 0.5, tf.int32)
    translation_x = tf.random.uniform([batch_size, 1], -shift[0], shift[0] + 1, dtype=tf.int32)
    translation_y = tf.random.uniform([batch_size, 1], -shift[1], shift[1] + 1, dtype=tf.int32)
    grid_x = tf.clip_by_value(
        tf.expand_dims(tf.range(image_size[0]), 0) + translation_x + 1, 0, image_size[0] + 1)
    grid_y = tf.clip_by_value(
        tf.expand_dims(tf.range(image_size[1]), 0) + translation_y + 1, 0, image_size[1] + 1)
    x = tf.pad(x, [[0, 0], [1, 1], [0, 0], [0, 0]])
    x = tf.gather(x, grid_x, batch_dims=1, axis=1)
    x = tf.pad(x, [[0, 0], [0, 0], [1, 1], [0, 0]])
    x = tf.gather(x, grid_y, batch_dims=1, axis=2)
    return x


def diff_augment(x, policy=""):
    """Apply the requested DiffAugment policy. Called on real and fake alike, never on the Generator output
    that gets saved or displayed."""
    if not policy:
        return x
    for op in policy.split(","):
        op = op.strip()
        if op == "translation":
            x = rand_translation(x)
    return x


def add_instance_noise(x, std_var):
    return x + tf.random.normal(tf.shape(x)) * std_var


def update_ema(ema_model, source_model, decay):
    """ema = ema*decay + source*(1-decay), in place."""
    ema_weights = ema_model.get_weights()
    source_weights = source_model.get_weights()
    ema_model.set_weights([e * decay + s * (1.0 - decay) for e, s in zip(ema_weights, source_weights)])


def set_lr(optimizer, value):
    try:
        optimizer.learning_rate.assign(value)
    except AttributeError:
        optimizer.learning_rate = value


def lr_factor(epoch, total_epochs, decay_from, floor):
    """1.0 until decay_from (a fraction of the stage), then linear down to `floor` at the last epoch."""
    start = decay_from * total_epochs
    if epoch <= start or decay_from >= 1.0:
        return 1.0
    span = max(1.0, total_epochs - start)
    return float(1.0 - min(1.0, (epoch - start) / span) * (1.0 - floor))


def instance_noise_at(epoch, total_epochs, initial_std, anneal_frac=0.6):
    """Linear anneal from initial_std down to 0 over the first `anneal_frac` of the stage."""
    if initial_std <= 0.0:
        return 0.0
    end = max(1.0, anneal_frac * total_epochs)
    return float(initial_std * max(0.0, 1.0 - (epoch - 1) / end))


print("Stabilizers defined (DiffAugment translation, instance noise, EMA, LR schedule).")

## 11. Training Step

### English
One `tf.function` per stage, built with that stage's flags baked in. Two things are worth reading closely:

**Where the augmentation sits.** `fake_images` is generated first and kept clean; only the *copies handed to
the Discriminator* get instance noise and DiffAugment. The Generator's gradient still flows back through
those copies, which is what makes DiffAugment work at all.

**The mode-seeking term** (`ms_weight > 0`, stage 3 only). The fake half-batches are built so that sample `i`
and sample `i + half` carry the **same class label** but different noise. `ms_signal` is then the average of
`|image_i - image_{i+half}| / |z_i - z_{i+half}|`: how much output moves per unit of input movement, within
one class. Subtracting it from the Generator loss rewards keeping that ratio high. Sixteen z values that all
map to the same cat face drive it toward zero, so the gradient actively pushes away from that state.
`ms_signal` is logged every epoch even when the weight is 0, so it is readable as a diagnostic in all stages.
Stage 3's `ms_weight` is set moderately (0.2): enough to keep pulling against collapse, but not so much
that it dominates the sharpening objective.

### Tiếng Việt
Mỗi stage một `tf.function`, được dựng với các cờ của stage đó nướng sẵn vào graph. Hai điểm đáng đọc kỹ:

**Chỗ đặt phép augment.** `fake_images` được sinh ra trước và giữ sạch; chỉ các *bản sao đưa cho
Discriminator* mới bị thêm instance noise và DiffAugment. Gradient của Generator vẫn chạy ngược qua các
bản sao đó -- chính điều này làm DiffAugment hoạt động được.

**Số hạng mode-seeking** (`ms_weight > 0`, chỉ stage 3). Nửa batch giả được dựng sao cho mẫu `i` và mẫu
`i + half` mang **cùng nhãn loài** nhưng noise khác nhau. `ms_signal` là trung bình của
`|ảnh_i - ảnh_{i+half}| / |z_i - z_{i+half}|`: đầu ra dịch chuyển bao nhiêu trên một đơn vị dịch chuyển
đầu vào, trong cùng một loài. Trừ nó khỏi loss của Generator tức là thưởng cho việc giữ tỷ lệ đó cao. Mười
sáu giá trị z cùng đổ về một khuôn mặt mèo sẽ kéo nó về 0, nên gradient chủ động đẩy ra khỏi trạng thái đó.
`ms_signal` vẫn được ghi log mỗi epoch kể cả khi trọng số bằng 0, để đọc như một chỉ số chẩn đoán ở mọi
stage. `ms_weight` của stage 3 được đặt ở mức vừa phải (0.2): đủ để tiếp tục kéo chống lại collapse, nhưng
không đến mức lấn át mục tiêu làm sắc nét.

In [ ]:
def make_train_step(generator, discriminator, gen_optimizer, disc_optimizer,
                    diffaug_policy="", ms_weight=0.0, noise_std_var=None, use_instance_noise=False):
    if noise_std_var is None:
        noise_std_var = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    paired_labels = ms_weight > 0.0

    def prepare_for_disc(images):
        """Only the copy handed to D is perturbed; the Generator's actual output stays untouched.
        Both branches are decided when the graph is built, so a stage with no noise and no DiffAugment
        (stage 1) runs exactly the plain recipe with no extra work per step."""
        if use_instance_noise:
            images = add_instance_noise(images, noise_std_var)
        return diff_augment(images, diffaug_policy)

    @tf.function
    def train_step(real_images, real_labels):
        batch_size = tf.shape(real_images)[0]
        half = batch_size // 2
        fake_bs = 2 * half

        noise = tf.random.normal([fake_bs, NOISE_DIM])
        if paired_labels:
            # sample i and sample i+half share a label, so the mode-seeking ratio measures
            # "different z, same class -> different face" and nothing else
            half_labels = tf.random.uniform([half], minval=0, maxval=NUM_CLASSES, dtype=tf.int32)
            fake_labels = tf.concat([half_labels, half_labels], axis=0)
        else:
            fake_labels = tf.random.uniform([fake_bs], minval=0, maxval=NUM_CLASSES, dtype=tf.int32)

        with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
            fake_images = generator([noise, fake_labels], training=True)

            real_in = prepare_for_disc(real_images)
            fake_in = prepare_for_disc(fake_images)

            real_logits = discriminator([real_in, real_labels], training=True)
            fake_logits = discriminator([fake_in, fake_labels], training=True)

            disc_loss = discriminator_loss(real_logits, fake_logits)
            adv_loss = generator_loss(fake_logits)

            image_gap = tf.reduce_mean(tf.abs(fake_images[:half] - fake_images[half:]), axis=[1, 2, 3])
            noise_gap = tf.reduce_mean(tf.abs(noise[:half] - noise[half:]), axis=1)
            ms_signal = tf.reduce_mean(image_gap / (noise_gap + 1e-6))

            gen_loss = adv_loss - ms_weight * ms_signal

        gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
        disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
        gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
        disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

        real_acc = tf.reduce_mean(tf.cast(tf.sigmoid(real_logits) > 0.5, tf.float32))
        fake_acc = tf.reduce_mean(tf.cast(tf.sigmoid(fake_logits) < 0.5, tf.float32))
        disc_acc = (real_acc + fake_acc) / 2.0

        return adv_loss, disc_loss, disc_acc, ms_signal

    return train_step


print("make_train_step() defined.")

## 12. Monitoring: Sample Grid and Per-Class Metrics

### English
**The sample grid** is 16 images per class, 8 columns x 4 rows, drawn into a 16 x 9 inch figure at 120 dpi
= exactly 1920x1080 pixels, so a folder of these can be turned into a 1080p timelapse with no rescaling.
Every panel is positioned by hand instead of by `tight_layout`, and the class name sits in the left margin,
so nothing can cover a label.

**Three metrics, computed separately for dogs and cats**, on a fixed set of noise vectors so the curve
reflects the model and not the sample:

- `struct` -- 1 minus the mean pairwise correlation of the generated images after each is converted to
  grayscale, downsampled to 32x32 and standardized. Standardizing removes brightness and colour, so what is
  left is *shape*. Sixteen identical faces in different colours score ~0.00; sixteen genuinely different
  faces score in the same range as real photos. **This is the collapse detector.**
- `color` -- the spread of per-image mean RGB. This catches the other failure mode, one narrow palette.
- `sharp` -- mean Sobel gradient magnitude, reported against the real-image reference as `sharp x`.
  Around 1.0 means generated edges are as strong as real ones; well under 1.0 means blur.

Real-image references for all three are computed once below and every stage reports its numbers relative
to them.

### Tiếng Việt
**Ảnh mẫu** là 16 ảnh mỗi loài, 8 cột x 4 hàng, vẽ trong figure 16 x 9 inch ở 120 dpi = đúng 1920x1080
pixel, nên cả thư mục ảnh này có thể ghép thành timelapse 1080p mà không phải resize. Mỗi ô được đặt tọa
độ thủ công thay vì để `tight_layout` tự xếp, và tên loài nằm ở lề trái, nên không còn cảnh chữ bị ảnh đè
lên.

**Ba chỉ số, tính riêng cho chó và mèo**, trên một bộ vector noise cố định để đường biểu diễn phản ánh
model chứ không phản ánh mẫu ngẫu nhiên:

- `struct` -- 1 trừ tương quan trung bình từng cặp giữa các ảnh sinh ra, sau khi chuyển xám, thu về 32x32
  và chuẩn hóa. Việc chuẩn hóa bỏ đi độ sáng và màu, thứ còn lại là *hình dạng*. Mười sáu khuôn mặt giống
  hệt nhau khác màu cho ~0.00; mười sáu khuôn mặt thật sự khác nhau cho giá trị ngang với ảnh thật.
  **Đây là bộ phát hiện collapse.**
- `color` -- độ phân tán của màu RGB trung bình mỗi ảnh. Chỉ số này bắt kiểu lỗi còn lại: chỉ một bảng
  màu hẹp.
- `sharp` -- độ lớn gradient Sobel trung bình, báo cáo theo tỷ lệ so với ảnh thật dạng `sharp x`. Gần 1.0
  nghĩa là cạnh của ảnh sinh ra mạnh ngang ảnh thật; thấp hơn 1.0 nhiều nghĩa là còn mờ.

Giá trị tham chiếu từ ảnh thật cho cả ba chỉ số được tính một lần ở dưới, và mọi stage đều báo cáo số
liệu của nó theo tỷ lệ với chúng.

In [ ]:
SAMPLES_PER_CLASS = 16
GRID_COLS = 8
ROWS_PER_CLASS = SAMPLES_PER_CLASS // GRID_COLS
GRID_ROWS = ROWS_PER_CLASS * NUM_CLASSES
FIG_W, FIG_H, FIG_DPI = 16.0, 9.0, 120  # -> 1920 x 1080 px, one 1080p video frame

FIXED_NOISE = tf.random.normal([SAMPLES_PER_CLASS * NUM_CLASSES, NOISE_DIM], seed=RANDOM_SEED)
FIXED_LABELS = tf.constant(
    [label for label in sorted(LABEL_TO_SPECIES) for _ in range(SAMPLES_PER_CLASS)], dtype=tf.int32)


def _grid_axes_rects(n_rows, n_cols, fig_w, fig_h, block_rows):
    """Exact axes rectangles (figure fractions) for a centered grid of square cells, plus the vertical
    center of each class block for its left-margin label. Hand-placed, so nothing can overlap."""
    left_margin, right_margin = 0.050, 0.012
    top_margin, bottom_margin = 0.085, 0.022
    gap = 0.055        # inches between cells
    block_gap = 0.28   # extra inches between the dog block and the cat block

    avail_w = fig_w * (1.0 - left_margin - right_margin)
    avail_h = fig_h * (1.0 - top_margin - bottom_margin) - block_gap
    cell = min((avail_w - (n_cols - 1) * gap) / n_cols,
               (avail_h - (n_rows - 1) * gap) / n_rows)

    grid_w = n_cols * cell + (n_cols - 1) * gap
    grid_h = n_rows * cell + (n_rows - 1) * gap + block_gap
    x0 = fig_w * left_margin + (avail_w - grid_w) / 2.0
    band_h = fig_h * (1.0 - top_margin - bottom_margin)
    y_top = fig_h * (1.0 - top_margin) - (band_h - grid_h) / 2.0

    rects, blocks = [], []
    for r in range(n_rows):
        extra = block_gap if r >= block_rows else 0.0
        y = y_top - (r + 1) * cell - r * gap - extra
        for c in range(n_cols):
            rects.append(((x0 + c * (cell + gap)) / fig_w, y / fig_h, cell / fig_w, cell / fig_h))
    for b in range(n_rows // block_rows):
        first, last = b * block_rows * n_cols, (b + 1) * block_rows * n_cols - 1
        blocks.append((0.5 * ((rects[first][1] + rects[first][3]) + rects[last][1]), x0 / fig_w))
    return rects, blocks


def save_sample_grid(model, epoch, save_dir, stage_label, model_label="G"):
    generated = model([FIXED_NOISE, FIXED_LABELS], training=False)
    generated = ((generated.numpy() + 1.0) * 127.5).clip(0, 255).astype("uint8")

    fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=FIG_DPI, facecolor="white")
    rects, blocks = _grid_axes_rects(GRID_ROWS, GRID_COLS, FIG_W, FIG_H, ROWS_PER_CLASS)
    for i, rect in enumerate(rects):
        ax = fig.add_axes(rect)
        ax.imshow(generated[i])
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    for b, (y_center, x_left) in enumerate(blocks):
        fig.text(x_left - 0.016, y_center, LABEL_TO_SPECIES[b].upper(), rotation=90,
                 va="center", ha="center", fontsize=15, fontweight="bold", color="#444444")
    fig.text(0.5, 0.955, f"{stage_label}  --  epoch {epoch}   ({model_label})",
             ha="center", va="center", fontsize=17, fontweight="bold")

    out_path = save_dir / f"epoch_{epoch:03d}.png"
    fig.savefig(out_path, dpi=FIG_DPI, facecolor="white")
    plt.close(fig)
    return out_path


print(f"Sample grid: {SAMPLES_PER_CLASS} per class, {GRID_COLS}x{GRID_ROWS}, "
      f"{int(FIG_W * FIG_DPI)}x{int(FIG_H * FIG_DPI)} px.")

In [ ]:
EVAL_SAMPLES_PER_CLASS = 64
EVAL_NOISE = tf.random.normal([EVAL_SAMPLES_PER_CLASS, NOISE_DIM], seed=RANDOM_SEED + 1)


def structure_diversity(images, size=32):
    """1 - mean pairwise correlation of grayscale, downsampled, per-image standardized images.
    ~0.0 = every sample has the same shape (mode collapse); higher = genuinely different shapes.
    Brightness and colour are normalized away, so this measures SHAPE only."""
    gray = tf.image.rgb_to_grayscale(images)
    gray = tf.image.resize(gray, [size, size], method="area")
    flat = tf.reshape(gray, [tf.shape(gray)[0], -1])
    flat = (flat - tf.reduce_mean(flat, axis=1, keepdims=True)) / (
        tf.math.reduce_std(flat, axis=1, keepdims=True) + 1e-6)
    n = tf.cast(tf.shape(flat)[0], tf.float32)
    d = tf.cast(tf.shape(flat)[1], tf.float32)
    gram = tf.matmul(flat, flat, transpose_b=True) / d
    off_diagonal = tf.reduce_sum(gram) - tf.linalg.trace(gram)
    return float(1.0 - off_diagonal / (n * (n - 1.0)))


def color_diversity(images):
    """Spread of per-image mean RGB across the batch. Low = one narrow palette."""
    per_image = tf.reduce_mean(images, axis=[1, 2])
    return float(tf.reduce_mean(tf.math.reduce_std(per_image, axis=0)))


def sharpness(images):
    """Mean Sobel gradient magnitude (unnormalized). Low = blurry."""
    gray = tf.image.rgb_to_grayscale((images + 1.0) / 2.0)
    edges = tf.image.sobel_edges(gray)
    return float(tf.reduce_mean(tf.sqrt(tf.reduce_sum(tf.square(edges), axis=-1) + 1e-8)))


def measure_images(images):
    return {"struct": structure_diversity(images),
            "color": color_diversity(images),
            "sharp": sharpness(images)}


def evaluate_generator(model):
    """Per-class metrics on the fixed evaluation noise."""
    out = {}
    for label, species in LABEL_TO_SPECIES.items():
        labels = tf.fill([EVAL_SAMPLES_PER_CLASS], label)
        images = model([EVAL_NOISE, labels], training=False)
        for key, value in measure_images(images).items():
            out[f"{key}_{species}"] = value
    return out


def _real_reference():
    refs = {}
    for species, label in SPECIES_TO_LABEL.items():
        subset = train_manifest[train_manifest["species"] == species].head(EVAL_SAMPLES_PER_CLASS * 2)
        ds = build_dataset(subset, LOCAL_IMAGES_DIR, EVAL_SAMPLES_PER_CLASS,
                           shuffle=False, augment_flag=False)
        images, _ = next(iter(ds))
        for key, value in measure_images(images).items():
            refs[f"{key}_{species}"] = value
    return refs


REAL_REFS = _real_reference()
print("Real-image references (the numbers every stage is compared against):")
for species in SPECIES_TO_LABEL:
    print(f"  {species:4s}  struct={REAL_REFS[f'struct_{species}']:.3f}  "
          f"color={REAL_REFS[f'color_{species}']:.3f}  sharp={REAL_REFS[f'sharp_{species}']:.4f}")

## 13. Checkpointing, Best-Model Tracking and the Collapse Guard

### English
Three separate things are saved, and they have different jobs:

1. **Rolling checkpoint** (`generator.weights.h5` etc., every `CHECKPOINT_EVERY` epochs) -- what a resume
   picks up from. Overwritten each time.
2. **Snapshots** (`snapshots/generator_e075.weights.h5`, every `snapshot_every` epochs) -- never overwritten.
   This is the rollback safety net.
3. **Best** (`best/generator_best.weights.h5`) -- the highest-scoring checkpoint of the stage, plus a small
   JSON recording which epoch it came from and why.

`quality_score` combines the metrics into one number in [0, 1]:

```
sharp_ratio = (generated sharpness) / (real sharpness), capped at 1.0
health      = min over classes of (generated struct) / (real struct)
score       = sharp_ratio * min(1.0, health / HEALTH_TARGET)
```

Capping `sharp_ratio` at 1.0 means over-sharpening into noise earns no bonus, and multiplying by the health
term means a sharp but collapsing model scores badly. So "best" means *sharpest model that is still diverse* --
and it does not have to be the final epoch. If quality peaks midway through a stage and drifts afterward,
`best/` is what gets exported, not whatever the last epoch happened to produce.

### Tiếng Việt
Ba thứ được lưu, mỗi thứ một nhiệm vụ khác nhau:

1. **Checkpoint quay vòng** (`generator.weights.h5` v.v., mỗi `CHECKPOINT_EVERY` epoch) -- cho việc
   resume. Bị ghi đè mỗi lần.
2. **Snapshot** (`snapshots/generator_e075.weights.h5`, mỗi `snapshot_every` epoch) -- không bao giờ bị
   ghi đè. Đây là lưới an toàn để quay về.
3. **Best** (`best/generator_best.weights.h5`) -- checkpoint điểm cao nhất của stage, kèm một file JSON
   nhỏ ghi lại nó ở epoch nào và vì sao.

`quality_score` gộp các chỉ số thành một số trong khoảng [0, 1] (công thức ở trên).

Việc chặn `sharp_ratio` ở 1.0 nghĩa là làm quá sắc thành nhiễu thì không được thưởng thêm, còn việc nhân
với số hạng sức khỏe nghĩa là một model sắc nét nhưng đang collapse sẽ bị điểm thấp. Vậy "best" nghĩa là
*model sắc nét nhất mà vẫn còn đa dạng* -- và không nhất thiết phải là epoch cuối cùng. Nếu chất lượng đạt
đỉnh giữa chừng một stage rồi trôi đi sau đó, `best/` mới là thứ được export, không phải bất cứ thứ gì
epoch cuối tạo ra.

In [ ]:
import json


def quality_score(metrics, refs=None):
    refs = refs or REAL_REFS
    gen_sharp = sum(metrics[f"sharp_{s}"] for s in SPECIES_TO_LABEL)
    real_sharp = sum(refs[f"sharp_{s}"] for s in SPECIES_TO_LABEL)
    sharp_ratio = min(1.0, gen_sharp / (real_sharp + 1e-8))
    health = min(metrics[f"struct_{s}"] / (refs[f"struct_{s}"] + 1e-8) for s in SPECIES_TO_LABEL)
    return float(sharp_ratio * min(1.0, health / HEALTH_TARGET)), float(health)


def save_rolling(model_dir, generator, discriminator, generator_ema=None):
    generator.save_weights(model_dir / "generator.weights.h5")
    discriminator.save_weights(model_dir / "discriminator.weights.h5")
    if generator_ema is not None:
        generator_ema.save_weights(model_dir / "generator_ema.weights.h5")


def save_snapshot(snap_dir, epoch, generator, generator_ema=None):
    generator.save_weights(snap_dir / f"generator_e{epoch:03d}.weights.h5")
    if generator_ema is not None:
        generator_ema.save_weights(snap_dir / f"generator_ema_e{epoch:03d}.weights.h5")


def save_best(best_dir, epoch, score, health, metrics, showcase_model):
    showcase_model.save_weights(best_dir / "generator_best.weights.h5")
    info = {"epoch": int(epoch), "score": round(float(score), 4), "health": round(float(health), 4)}
    info.update({k: round(float(v), 4) for k, v in metrics.items()})
    (best_dir / "best_info.json").write_text(json.dumps(info, indent=2))
    return info


def load_best_info(best_dir):
    path = best_dir / "best_info.json"
    if path.exists():
        try:
            return json.loads(path.read_text())
        except json.JSONDecodeError:
            return None
    return None


print("Checkpoint helpers defined.")

## 14. The Stage Runner

### English
One function runs any stage. It resumes from its own folder if there is one, otherwise it loads the previous
stage's weights (both Generator **and** Discriminator -- starting stage 2 with a fresh D would destroy
everything stage 1 learned in a few hundred steps).

Nothing here stops training on its own except stage 3's collapse guard, and even that leaves every file in
place. The printed warnings are signals for you to look at the grid, not automatic decisions.

Each epoch prints one line:

```
[stage3] epoch 42/100 | gen=1.52 disc=0.88 acc=0.84 ms=0.20 | struct d/c 0.61/0.58 (health 0.79)
         color d/c 0.19/0.17 | sharp 0.83x | score 0.744 (best 0.761 @ e30) | 38.2s
```

### Tiếng Việt
Một hàm chạy được mọi stage. Nó resume từ thư mục của chính nó nếu có, không thì nạp trọng số của stage
trước (cả Generator **và** Discriminator -- bắt đầu stage 2 với một D mới tinh sẽ phá hủy mọi thứ stage 1
học được chỉ trong vài trăm bước).

Không có gì ở đây tự động dừng train, trừ bộ phát hiện collapse của stage 3, và ngay cả nó cũng giữ nguyên
mọi file. Các cảnh báo in ra là tín hiệu để bạn mở ảnh lên xem, không phải quyết định tự động.

In [ ]:
import time


def run_stage(stage_key, epochs=None):
    cfg = dict(STAGES[stage_key])
    total_epochs = int(epochs or cfg["epochs"])

    model_dir = MODELS_DIR / MODEL_SUBDIR / stage_key
    figures_dir = FIGURES_DIR / MODEL_SUBDIR / stage_key
    snap_dir = model_dir / "snapshots"
    best_dir = model_dir / "best"
    for directory in (model_dir, figures_dir, snap_dir, best_dir):
        directory.mkdir(parents=True, exist_ok=True)

    history_path = model_dir / "history.csv"
    gen_path = model_dir / "generator.weights.h5"
    disc_path = model_dir / "discriminator.weights.h5"
    ema_path = model_dir / "generator_ema.weights.h5"

    use_ema = cfg["ema_decay"] > 0.0
    generator = build_generator()
    discriminator = build_discriminator(name=f"discriminator_{stage_key}")
    generator_ema = build_generator(name="generator_ema") if use_ema else None
    gen_optimizer, disc_optimizer = make_optimizers(cfg["gen_lr"], cfg["disc_lr"])
    noise_std_var = tf.Variable(0.0, trainable=False, dtype=tf.float32)
    train_step = make_train_step(generator, discriminator, gen_optimizer, disc_optimizer,
                                 diffaug_policy=cfg["diffaug"], ms_weight=cfg["ms_weight"],
                                 noise_std_var=noise_std_var,
                                 use_instance_noise=cfg["instance_noise"] > 0.0)

    # ---- where do the starting weights come from? ----
    start_epoch, history = 1, []
    if history_path.exists() and gen_path.exists() and disc_path.exists():
        history_df = pd.read_csv(history_path)
        history = history_df.to_dict("records")
        start_epoch = int(history_df["epoch"].max()) + 1
        generator.load_weights(gen_path)
        discriminator.load_weights(disc_path)
        if use_ema:
            if ema_path.exists():
                generator_ema.load_weights(ema_path)
            else:
                generator_ema.set_weights(generator.get_weights())
        print(f"[{stage_key}] Resuming from epoch {start_epoch} (found this stage's checkpoint).")
    elif cfg["init_from"]:
        prev_dir = MODELS_DIR / MODEL_SUBDIR / cfg["init_from"]
        prev_ema = prev_dir / "generator_ema.weights.h5"
        prev_gen = prev_ema if (cfg["init_from_ema"] and prev_ema.exists()) else prev_dir / "generator.weights.h5"
        prev_disc = prev_dir / "discriminator.weights.h5"
        assert prev_gen.exists() and prev_disc.exists(), (
            f"[{stage_key}] needs {cfg['init_from']} to be trained first -- "
            f"missing {prev_gen if not prev_gen.exists() else prev_disc}")
        generator.load_weights(prev_gen)
        discriminator.load_weights(prev_disc)
        if use_ema:
            generator_ema.set_weights(generator.get_weights())
        print(f"[{stage_key}] Starting from {cfg['init_from']} weights ({prev_gen.name} + {prev_disc.name}).")
    else:
        print(f"[{stage_key}] Starting fresh (random initialization).")
        if use_ema:
            generator_ema.set_weights(generator.get_weights())

    if start_epoch > total_epochs:
        print(f"[{stage_key}] Already at epoch {start_epoch - 1}/{total_epochs} -- nothing to do.")
        return generator_ema if use_ema else generator, discriminator, pd.DataFrame(history)

    best_info = load_best_info(best_dir)
    best_score = float(best_info["score"]) if best_info else -1.0
    best_epoch = int(best_info["epoch"]) if best_info else -1
    collapse_hits = 0
    showcase = generator_ema if use_ema else generator
    showcase_label = "EMA" if use_ema else "G"

    print(f"[{stage_key}] {total_epochs} epochs | G lr={cfg['gen_lr']:.1e} D lr={cfg['disc_lr']:.1e} | "
          f"noise={cfg['instance_noise']} diffaug='{cfg['diffaug']}' ema={cfg['ema_decay']} "
          f"ms={cfg['ms_weight']} lr_floor={cfg['lr_floor']} | sampling the {showcase_label} model")

    for epoch in range(start_epoch, total_epochs + 1):
        epoch_start = time.time()
        factor = lr_factor(epoch, total_epochs, cfg["lr_decay_from"], cfg["lr_floor"])
        set_lr(gen_optimizer, cfg["gen_lr"] * factor)
        set_lr(disc_optimizer, cfg["disc_lr"] * factor)
        noise_std_var.assign(instance_noise_at(epoch, total_epochs, cfg["instance_noise"]))

        gen_losses, disc_losses, disc_accs, ms_values = [], [], [], []
        for real_images, real_labels in train_ds:
            gen_loss, disc_loss, disc_acc, ms_signal = train_step(real_images, real_labels)
            gen_losses.append(float(gen_loss))
            disc_losses.append(float(disc_loss))
            disc_accs.append(float(disc_acc))
            ms_values.append(float(ms_signal))

        if use_ema:
            update_ema(generator_ema, generator, cfg["ema_decay"])

        metrics = evaluate_generator(showcase)
        score, health = quality_score(metrics)
        epoch_time = time.time() - epoch_start
        sharp_ratio = (sum(metrics[f"sharp_{s}"] for s in SPECIES_TO_LABEL)
                       / sum(REAL_REFS[f"sharp_{s}"] for s in SPECIES_TO_LABEL))

        row = {"stage": stage_key, "epoch": epoch,
               "gen_loss": float(np.mean(gen_losses)), "disc_loss": float(np.mean(disc_losses)),
               "disc_acc": float(np.mean(disc_accs)), "ms_signal": float(np.mean(ms_values)),
               "score": score, "health": health, "sharp_ratio": float(sharp_ratio),
               "lr_factor": factor, "instance_noise": float(noise_std_var.numpy()),
               "seconds": epoch_time}
        row.update({k: float(v) for k, v in metrics.items()})
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        print(f"[{stage_key}] epoch {epoch}/{total_epochs} | "
              f"gen={row['gen_loss']:.3f} disc={row['disc_loss']:.3f} acc={row['disc_acc']:.3f} "
              f"ms={row['ms_signal']:.3f} | struct d/c {metrics['struct_dog']:.3f}/{metrics['struct_cat']:.3f} "
              f"(health {health:.2f}) color d/c {metrics['color_dog']:.3f}/{metrics['color_cat']:.3f} | "
              f"sharp {sharp_ratio:.2f}x | score {score:.3f} "
              f"(best {'--' if best_epoch < 0 else f'{best_score:.3f} @ e{best_epoch}'}) | {epoch_time:.1f}s")

        if row["disc_acc"] > 0.95:
            print("  WARNING: Discriminator accuracy above 0.95 -- it may be overpowering the Generator.")
        elif row["disc_acc"] < 0.55:
            print("  WARNING: Discriminator accuracy below 0.55 -- it may have collapsed.")
        for species in SPECIES_TO_LABEL:
            ratio = metrics[f"struct_{species}"] / (REAL_REFS[f"struct_{species}"] + 1e-8)
            if ratio < COLLAPSE_STOP_RATIO:
                print(f"  WARNING: {species} shape diversity is {ratio:.2f}x the real reference -- "
                      f"the {species} samples may be converging on one face. Open the grid and check.")

        if epoch % CHECKPOINT_EVERY == 0 or epoch == total_epochs:
            save_rolling(model_dir, generator, discriminator, generator_ema)
            grid_path = save_sample_grid(showcase, epoch, figures_dir,
                                         f"{MODEL_SUBDIR} {stage_key}", showcase_label)
            print(f"  Checkpoint saved. Sample grid: {grid_path} -- look at it before trusting any number above.")
            if score > best_score:
                best_score, best_epoch = score, epoch
                save_best(best_dir, epoch, score, health, metrics, showcase)
                print(f"  New best model for this stage (score {score:.3f}) -> {best_dir/'generator_best.weights.h5'}")

        if epoch % cfg["snapshot_every"] == 0:
            save_snapshot(snap_dir, epoch, generator, generator_ema)
            print(f"  Permanent snapshot kept: {snap_dir}/generator_e{epoch:03d}.weights.h5")

        if cfg["early_stop"]:
            collapse_hits = collapse_hits + 1 if health < COLLAPSE_STOP_RATIO else 0
            if collapse_hits >= COLLAPSE_PATIENCE:
                save_rolling(model_dir, generator, discriminator, generator_ema)
                save_snapshot(snap_dir, epoch, generator, generator_ema)
                print(f"\n[{stage_key}] EARLY STOP at epoch {epoch}: shape diversity stayed below "
                      f"{COLLAPSE_STOP_RATIO:.2f}x the real reference for {COLLAPSE_PATIENCE} checks in a row.")
                print(f"[{stage_key}] Use the best model instead: {best_dir/'generator_best.weights.h5'} "
                      f"(epoch {best_epoch}, score {best_score:.3f}).")
                break

    print(f"\n[{stage_key}] Done. Best model: epoch {best_epoch}, score {best_score:.3f} -> "
          f"{best_dir/'generator_best.weights.h5'}")
    return showcase, discriminator, pd.DataFrame(history)


print("run_stage() defined.")

## 15. Stage 1 -- Find the Structure (150 epochs)

### English
The fast, plain recipe: equal learning rate for both networks, no extra regularization, run only as far as
it is useful for finding structure. Expect roughly 37-40 s per epoch, so about 95 minutes.

**This starts from scratch**, initializing fresh Generator and Discriminator weights. Stage 2 and stage 3
each build directly on whatever this stage produces.

What to watch: cats should look converged around epoch 100-120, dogs should be close behind by 150.
`disc_acc` will sit high (0.85-0.95) and climb further late in the stage -- that is expected here and is
exactly why the stage stops at 150, handing off to stage 2 before the Discriminator can fully overpower the
Generator.

### Tiếng Việt
Công thức nhanh, thuần túy: LR bằng nhau cho cả hai mạng, không thêm regularization, chỉ chạy đến mức còn
có ích cho việc tìm cấu trúc. Ước tính 37-40 giây mỗi epoch, tức khoảng 95 phút.

**Stage này bắt đầu từ đầu**, khởi tạo trọng số Generator và Discriminator hoàn toàn mới. Stage 2 và
stage 3 sẽ xây trực tiếp trên những gì stage này tạo ra.

Cần theo dõi: mèo sẽ hội tụ khoảng epoch 100-120, chó theo sát đến 150. `disc_acc` sẽ ở mức cao
(0.85-0.95) và còn tăng thêm vào cuối stage -- điều đó là bình thường ở đây và chính là lý do stage dừng
ở 150, bàn giao cho stage 2 trước khi Discriminator có thể áp đảo hoàn toàn Generator.

In [ ]:
gen_s1, disc_s1, hist_s1 = run_stage("stage1")

## 16. Stage 2 -- Refine Slowly and Safely (200 epochs)

### English
TTUR (D held to a third of G's learning rate), instance noise annealed over the first 60% of the stage,
DiffAugment translation, and EMA, run for 200 epochs. The LR schedule fractions (`lr_decay_from=0.7`,
`lr_floor=0.4`) mean the taper only starts at epoch 140 and the floor is reached at epoch 200 -- a long
full-strength phase followed by a gentle taper.

Progress here is *supposed* to look slower than stage1 -- there is simply more time being spent per unit of
improvement, which is the point of a long, cautious refinement stage. Watch `struct_dog` / `struct_cat` for
continued stability (they should not need to drop for `score` to keep climbing) and `disc_acc` for staying
comfortably under 0.95 all the way to epoch 200.

### Tiếng Việt
TTUR (LR của D giữ ở 1/3 của G), instance noise giảm dần trong 60% đầu stage, DiffAugment translation, và
EMA, chạy trong 200 epoch. Tỷ lệ phần trăm của lịch LR (`lr_decay_from=0.7`, `lr_floor=0.4`) nghĩa là
taper chỉ bắt đầu ở epoch 140 và chạm sàn ở epoch 200 -- một pha giữ LR đầy đủ dài, theo sau là taper nhẹ
nhàng.

Tiến độ ở đây *đáng lẽ* phải trông chậm hơn stage1 -- đơn giản là có nhiều thời gian hơn cho mỗi đơn vị
cải thiện, đúng như mục đích của một stage tinh chỉnh dài và thận trọng. Theo dõi `struct_dog` /
`struct_cat` xem có giữ ổn định không (không cần tụt xuống để `score` tiếp tục tăng), và `disc_acc` xem
có giữ dưới 0.95 thoải mái suốt đến epoch 200 hay không.

In [ ]:
gen_s2, disc_s2, hist_s2 = run_stage("stage2")

## 17. Stage 3 -- Sharpen and Hold (up to 100 epochs, stops early on collapse)

### English
Low LR, instance noise off, DiffAugment translation, EMA, and the mode-seeking term switched on
(`ms_weight=0.2`). This is the polishing pass: the goal is sharper output while holding onto the structure
and diversity that stage 1 and stage 2 already built.

This stage does not fully trust itself to keep improving for the whole 100 epochs, so three safety
mechanisms are active throughout: permanent snapshots every 10 epochs, a scored `best/` checkpoint updated
at every checkpoint interval, and an early stop after two consecutive collapse readings. If quality peaks
early and drifts afterward, `best/` -- not the final epoch -- is what should be exported.

Watch `health` (should stay well above the collapse threshold), `disc_acc` (should stay controlled, not
climb steadily toward 0.95+), and `score` in the printed log each checkpoint -- and look at the sample grid
itself, not just the numbers.

Export `stage3/best/generator_best.weights.h5` in `06_export_model.ipynb`, not necessarily the final epoch.

### Tiếng Việt
LR thấp, tắt instance noise, DiffAugment translation, EMA, và bật số hạng mode-seeking (`ms_weight=0.2`).
Đây là pha đánh bóng: mục tiêu là đầu ra sắc nét hơn trong khi vẫn giữ được cấu trúc và độ đa dạng mà
stage 1 và stage 2 đã xây dựng.

Stage này không hoàn toàn tin rằng nó sẽ cải thiện liên tục suốt 100 epoch, nên ba cơ chế an toàn luôn
hoạt động: snapshot vĩnh viễn mỗi 10 epoch, checkpoint `best/` có chấm điểm cập nhật ở mỗi mốc checkpoint,
và tự dừng sau hai lần đo liên tiếp báo collapse. Nếu chất lượng đạt đỉnh sớm rồi trôi đi sau đó, `best/`
-- chứ không phải epoch cuối -- mới là thứ nên được export.

Theo dõi `health` (nên giữ ở mức cao hơn hẳn ngưỡng collapse), `disc_acc` (nên được kiểm soát, không tăng
đều về phía 0.95+), và `score` trong log in ra ở mỗi checkpoint -- và nhìn vào chính ảnh mẫu, không chỉ
các con số.

Export `stage3/best/generator_best.weights.h5` ở `06_export_model.ipynb`, không nhất thiết là epoch cuối.

In [ ]:
gen_s3, disc_s3, hist_s3 = run_stage("stage3")

## 18. Review All Three Stages Together

### English
The three history files are concatenated so the whole run reads as one timeline. The second panel is the one
that matters most: per-class `struct` against the real-image reference. A line drifting down toward the
dashed collapse threshold is the early warning signal.

### Tiếng Việt
Ba file lịch sử được nối lại để cả quá trình đọc như một đường thời gian duy nhất. Bảng thứ hai là bảng
quan trọng nhất: `struct` theo từng loài so với tham chiếu ảnh thật. Một đường tụt dần xuống ngưỡng
collapse (nét đứt) chính là tín hiệu cảnh báo sớm.

In [ ]:
frames = []
offset = 0
for key in STAGES:
    path = MODELS_DIR / MODEL_SUBDIR / key / "history.csv"
    if path.exists():
        df = pd.read_csv(path)
        df["global_epoch"] = df["epoch"] + offset
        offset = df["global_epoch"].max()
        frames.append(df)

if not frames:
    print("No history yet -- run at least one stage first.")
else:
    all_hist = pd.concat(frames, ignore_index=True)
    boundaries = all_hist.groupby("stage")["global_epoch"].min().sort_values().tolist()[1:]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(all_hist["global_epoch"], all_hist["gen_loss"], label="Generator")
    axes[0].plot(all_hist["global_epoch"], all_hist["disc_loss"], label="Discriminator")
    axes[0].set_title("Losses")
    axes[0].set_xlabel("epoch (all stages)")
    axes[0].legend(fontsize=8)

    for species, colour in zip(SPECIES_TO_LABEL, ["tab:blue", "tab:orange"]):
        axes[1].plot(all_hist["global_epoch"], all_hist[f"struct_{species}"], color=colour, label=species)
        axes[1].axhline(REAL_REFS[f"struct_{species}"], color=colour, alpha=0.35, lw=1)
        axes[1].axhline(REAL_REFS[f"struct_{species}"] * COLLAPSE_STOP_RATIO, color=colour,
                        alpha=0.6, lw=1, ls="--")
    axes[1].set_title("Shape diversity vs real (solid = real, dashed = collapse threshold)")
    axes[1].set_xlabel("epoch (all stages)")
    axes[1].legend(fontsize=8)

    axes[2].plot(all_hist["global_epoch"], all_hist["sharp_ratio"], label="sharpness / real")
    axes[2].plot(all_hist["global_epoch"], all_hist["score"], label="quality score")
    axes[2].plot(all_hist["global_epoch"], all_hist["disc_acc"], label="disc_acc", alpha=0.6)
    axes[2].set_title("Sharpness, score, D accuracy")
    axes[2].set_xlabel("epoch (all stages)")
    axes[2].legend(fontsize=8)

    for ax in axes:
        for boundary in boundaries:
            ax.axvline(boundary, color="grey", lw=1, ls=":")
        ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "05_all_stages.png", dpi=150)
    plt.show()

    for key in STAGES:
        info = load_best_info(MODELS_DIR / MODEL_SUBDIR / key / "best")
        if info:
            print(f"{key}: best epoch {info['epoch']}, score {info['score']}, health {info['health']}")

In [ ]:
from PIL import Image as PILImage

latest_stage = None
for key in STAGES:
    if (FIGURES_DIR / MODEL_SUBDIR / key).exists() and any((FIGURES_DIR / MODEL_SUBDIR / key).glob("epoch_*.png")):
        latest_stage = key

if latest_stage is None:
    print("No sample grids yet.")
else:
    grids = sorted((FIGURES_DIR / MODEL_SUBDIR / latest_stage).glob("epoch_*.png"))
    latest = grids[-1]
    fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(PILImage.open(latest))
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Showing {latest_stage} / {latest.name} ({len(grids)} grids saved in this stage).")

## Summary

### English

**What this notebook does.** Trains `sobelv5` in three stages sharing one architecture: 150 epochs of a
fast plain-DCGAN recipe to find face structure, 200 epochs of TTUR-restrained refinement, then up to 100
epochs of low-LR sharpening with a moderate mode-seeking term (`ms_weight=0.2`) and a LR floor that keeps
some learning signal alive to the end of the stage (`lr_floor=0.3`).

**Why it is built this way.** Splitting training into three stages with different jobs lets each one use
the recipe that suits it: an aggressive, unregularized recipe finds structure fastest, a heavily-restrained
recipe refines safely without the Discriminator running away, and a final low-LR pass sharpens while
actively guarding against mode collapse. Both dog and cat are trained with no per-class weighting, so
neither species is deliberately favored.

**Monitoring.** The 16-per-class 1920x1080 sample grid, the per-class `struct`/`color`/`sharp` metrics and
their real-image references, and the three-tier checkpoint system (rolling / permanent snapshots / scored
best) all apply throughout every stage.

**What to do while it runs.** Look at the grids. The warnings are prompts to look, not verdicts.

**Next.** `05_evaluation_report.ipynb` for the evaluation, `06_export_model.ipynb` to export -- export
`stage3/best/generator_best.weights.h5`, not necessarily the final epoch.

### Tiếng Việt

**Notebook này làm gì.** Train `sobelv5` qua ba stage dùng chung một kiến trúc: 150 epoch công thức
DCGAN thuần chạy nhanh để tìm cấu trúc khuôn mặt, 200 epoch tinh chỉnh có kìm hãm bằng TTUR, rồi tối đa
100 epoch làm rõ nét với LR thấp, dùng số hạng mode-seeking vừa phải (`ms_weight=0.2`) và sàn LR giữ
một chút tín hiệu học sống đến hết stage (`lr_floor=0.3`).

**Vì sao làm theo cách này.** Tách train thành ba stage với nhiệm vụ khác nhau để mỗi stage dùng đúng
công thức phù hợp: một công thức mạnh dạn, không regularization tìm cấu trúc nhanh nhất, một công thức bị
kìm hãm mạnh tinh chỉnh an toàn mà không để Discriminator chạy mất kiểm soát, và một pha LR thấp cuối cùng
làm sắc nét trong khi chủ động canh chừng mode collapse. Cả chó và mèo đều được train không có trọng số ưu
tiên loài nào, nên không loài nào được ưu ái có chủ đích.

**Theo dõi.** Ảnh mẫu 16 ảnh/loài khung 1920x1080, các chỉ số `struct`/`color`/`sharp` riêng từng loài và
tham chiếu từ ảnh thật, cùng hệ thống checkpoint ba tầng (quay vòng / snapshot vĩnh viễn / best có chấm
điểm) đều áp dụng xuyên suốt mọi stage.

**Trong lúc train thì làm gì.** Nhìn ảnh. Các cảnh báo là lời nhắc để nhìn, không phải kết luận.

**Tiếp theo.** `05_evaluation_report.ipynb` để đánh giá, `06_export_model.ipynb` để export -- export
`stage3/best/generator_best.weights.h5`, không nhất thiết là epoch cuối.